# NUTDTS 816 · Assignment 1: Exploratory analysis and decomposition (7%)

Due Monday 21 September 2026, 23:59 WAT. Series A: nigeria_cpi. Series B: Austourists.

**Name / Group:** Fregene Ofe Favour  **Date:** 20-09-2026

This notebook must run top to bottom from a fresh Colab runtime. Keep the section headings; put your commentary in the markdown cells and your code in the code cells. Finish with the AI-use statement.

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/Favour-Fregene/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "austourists"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

## 1. Load and tidy (DatetimeIndex, frequency, gaps)

*tsdata loaders; asfreq; explain any gap handling*

In [ ]:
cpi = tsdata.nigeria_cpi()      # Fetch the Cpi data
print(type(cpi), cpi.index.freq)
print(cpi.head(6))
print(cpi.index[:3])

print()
print('-' * 40)
print()

at = tsdata.austourists()      # Fetch the austourists data
print(type(at), at.index.freq)
print(at.head(6))
print(at.index[:3])


In [ ]:
#Check for duplicate for both data
at_duplicate = at.index.duplicated().sum()
print("Total A.T Duplicate:", at_duplicate)

cpi_duplicate = cpi.index.duplicated().sum()
print("Total CPI Duplicate:", cpi_duplicate)

In [ ]:
#Check for missing values for both data
at_isnull = at.isnull().sum()
print("Total A.T missing value:", at_isnull)

cpi_isnull = cpi.isnull().sum()
print("Total CPI missing value:", cpi_isnull)

Commentary:

## 2. Time, seasonal, subseries and lag plots for each series

*one figure per plot type; annotate what each shows*

In [ ]:
# Time Plot for both Series
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
at.plot(ax=axes[0], title='austourists', ylabel= 'Tourists', xlabel= 'Year')
cpi.plot(ax=axes[1], title='Nigeria CPI, monthly Time plot', ylabel='CPI', xlabel='Year')
for ax in axes.flat: ax.set_xlabel('')


In [ ]:
# Seasonal Plot for Austourists
def seasonal_plot(s, period_label='year', title=''):
    df = pd.DataFrame({'v': s.values, 'year': s.index.year, 'month': s.index.month})
    piv = df.pivot(index='month', columns='year', values='v')
    ax = piv.plot(legend=False, colormap='viridis', title=title, figsize=(8, 3.4))
    ax.set_xlabel('Month'); ax.set_xticks(range(1, 13))
    for col in piv.columns[-1:]: ax.annotate(str(col), (12, piv[col].iloc[-1]), fontsize=8)
    return ax


seasonal_plot(at, title='Seasonal plot: austourists')
plt.show()

In [ ]:
# Seasonal Plot for Nigeria CPI
seasonal_plot(cpi, title='Seasonal plot: Nigeria cpi ')
plt.show()

In [ ]:
# Seasonal Subseries plot for both serie
def subseries_plot(s, title=''):
    df = pd.DataFrame({'v': s.values, 'year': s.index.year, 'month': s.index.month})
    fig, axes = plt.subplots(1, 12, figsize=(10, 3), sharey=True)
    for m, ax in zip(range(1, 13), axes):
        sub = df[df.month == m]
        ax.plot(sub.year, sub.v, lw=1); ax.axhline(sub.v.mean(), color='#B8860B', lw=1.2)
        ax.set_title(['J','F','M','A','M','J','J','A','S','O','N','D'][m-1]); ax.set_xticks([]); ax.grid(False)
    fig.suptitle(title, y=1.02); return fig

subseries_plot(cpi, title = 'Seasonal subseries plot: Nigeria CPI')
plt.show()
print()
subseries_plot(at, title = 'Seasonal subseries plot: Austourists')
plt.show()

In [ ]:
# Lag plot for Austourists
def lag_plot_at(s, lags=(1, 2, 3, 6, 12, 24), title=''):
    plt.close('all')
    s = s.squeeze()
    fig, axes = plt.subplots(2, 3, figsize=(9, 5.5))
    for k, ax in zip(lags, axes.flat):
        ax.scatter(s.shift(k), s, s=8)
        ax.set_title(f'lag {k}')
        ax.set_xlabel(f'x(t-{k})')
        ax.set_ylabel('x(t)')
    fig.suptitle(title, y=1.0);
    plt.tight_layout()
    return fig

lag_plot_at(np.log(at), title='Lag plots of Austourists')
plt.show()

In [ ]:
# Lag plot for Nigeria CPI
def lag_plot_cpi(s, lags=(1, 2, 3, 6, 12, 24), title=''):
    plt.close('all')
    s = s.squeeze()
    fig, axes = plt.subplots(2, 3, figsize=(9, 5.5))
    for k, ax in zip(lags, axes.flat):
        ax.scatter(s.shift(k), s, s=8)
        ax.set_title(f'lag {k}')
        ax.set_xlabel(f'x(t-{k})')
        ax.set_ylabel('x(t)')
    fig.suptitle(title, y=1.0);
    plt.tight_layout()
    return fig
lag_plot_cpi(np.log(cpi), title='Lag plots of Nigeria CPI')
plt.show()

Commentary:

## 3. ACF of each series and of its first difference

*plot_acf; interpret trend, seasonality, dependence*

In [ ]:
# ACF and First difference of austourists
from statsmodels.graphics.tsaplots import plot_acf
plt.close('all')
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
plot_acf(np.log(at), lags=36, ax=axes[0], title='Austourists - Autocorrelation Fuction');
plot_acf(np.log(at).diff().dropna(), lags=36, ax=axes[1], title='Austourists - First Difference')
plt.tight_layout()
plt.show()

In [ ]:
# ACF and First difference of Nigeria CPI
from statsmodels.graphics.tsaplots import plot_acf
plt.close('all')
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
plot_acf(np.log(cpi), lags=36, ax=axes[0], title='Nigeria CPI - Autocorrelation Fuction');
plot_acf(np.log(cpi).diff().dropna(), lags=36, ax=axes[1], title='Nigeria CPI - First Difference')
plt.tight_layout()
plt.show()

Commentary:

## 4. Additive or multiplicative? Justify

*evidence from the plots and the log plot*

In [ ]:
# For Austourists
from statsmodels.tsa.seasonal import seasonal_decompose
import numpy as np
plt.close('all')

add = seasonal_decompose(at, model='additive', period=4)
mul = seasonal_decompose(at, model='multiplicative', period=4)

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
add.resid.plot(ax=axes[0], title='Additive residuals')
mul.resid.plot(ax=axes[1], title='Multiplicative residuals')

np.log(at).plot(title='Log of austourists')
plt.tight_layout()
plt.show()

In [ ]:
# For Nigeria CPI
from statsmodels.tsa.seasonal import seasonal_decompose
import numpy as np
plt.close('all')

add = seasonal_decompose(cpi, model='additive', period=12)
mul = seasonal_decompose(cpi, model='multiplicative', period=12)

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
add.resid.plot(ax=axes[0], title='Additive residuals')
mul.resid.plot(ax=axes[1], title='Multiplicative residuals')

np.log(cpi).plot(title='Log of Nigeria CPI')
plt.tight_layout()
plt.show()

Commentary:

•	Austourists data is multiplicative. The seasoanl swings widen as the level rises, the log transform even them out, and the multiplicative residuals are tigter than  the additive ones..


•	Nigeria CPI data is multiplicative. The trend accelerates, the log makes growth near linear, and the multiplicative residuals are small and centred on 1. Seasonality is negligible.

## 5. Classical and STL decomposition compared

*seasonal_decompose vs STL; where do they disagree and why*

In [ ]:
#Classical Decompostion
from statsmodels.tsa.seasonal import seasonal_decompose
dec_add  = seasonal_decompose(cpi, model='multiplicative', period=12)
dec_mult = seasonal_decompose(at,   model='multiplicative', period=4)

fig, axes = plt.subplots(4, 2, figsize=(11, 8), sharex='col')
for col, (dec, name) in enumerate([(dec_add, 'Nigeria CPI  (multiplicative)'), (dec_mult, 'Austourists (multiplicative)')]):
    for row, comp in enumerate(['observed', 'trend', 'seasonal', 'resid']):
        getattr(dec, comp).plot(ax=axes[row, col], lw=1); axes[row, col].set_ylabel(comp); axes[row, col].set_xlabel('')
    axes[0, col].set_title(name)


In [ ]:
# STL: Seasonal-Trend decomposition using LOESS(austourists)
from statsmodels.tsa.seasonal import STL
stl_at = STL(np.log(at), period=12, seasonal=13, robust=True).fit()
fig = stl_at.plot(); fig.set_size_inches(9, 7)

In [ ]:
# STL: Seasonal-Trend decomposition using LOESS(nigeria cpi)
from statsmodels.tsa.seasonal import STL
stl_cpi = STL(np.log(cpi), period=12, seasonal=13, robust=True).fit()
fig = stl_cpi.plot(); fig.set_size_inches(9, 7)

Commentary:

## 6. Transformation decision

*log / Box-Cox; show the effect*

In [ ]:
#Transformation of the Austourists data
from scipy import stats

lam = stats.boxcox_normmax(at.values, method='mle')
print(f'MLE Box-Cox lambda for austourists: {lam:.3f}  (0 = log)')
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
at.plot(ax=axes[0], title='austourists: original (λ = 1)')
np.log(at).plot(ax=axes[1], title='austourists: log (λ = 0)')
pd.Series(stats.boxcox(at.values, lmbda=lam), index=at.index).plot(ax=axes[2], title=f'austourists: Box-Cox (λ = {lam:.2f})')
for ax in axes: ax.set_xlabel('')

plt.tight_layout()
plt.show()


In [ ]:
#Transformation of the Nigeria CPI data
from scipy import stats
lam = stats.boxcox_normmax(cpi.values, method='mle')
print(f'MLE Box-Cox lambda for nigeria cpi: {lam:.3f}  (0 = log)')
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
cpi.plot(ax=axes[0], title='nigeria cpi: original (λ = 1)')
np.log(cpi).plot(ax=axes[1], title='nigeria cpi: log (λ = 0)')
pd.Series(stats.boxcox(cpi.values, lmbda=lam), index=cpi.index).plot(ax=axes[2], title=f'nigeria cpi: Box-Cox (λ = {lam:.2f})')
for ax in axes: ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
#Classical Decompostion After Transformation
from statsmodels.tsa.seasonal import seasonal_decompose
at_bc  = pd.Series(stats.boxcox(at.values,  lmbda=0.082),  index=at.index)
cpi_bc = pd.Series(stats.boxcox(cpi.values, lmbda=-0.429), index=cpi.index)

cases = [('austourists', at, np.log(at), 'log', 4),
         ('Nigeria CPI', cpi, cpi_bc, 'Box-Cox', 12)]

fig, axes = plt.subplots(2, 2, figsize=(11, 5))
for i, (name, raw, tr, tlabel, p) in enumerate(cases):
    seasonal_decompose(raw, model='additive', period=p).resid.plot(
        ax=axes[i, 0], title=f'{name}: raw, additive residuals')
    seasonal_decompose(tr, model='additive', period=p).resid.plot(
        ax=axes[i, 1], title=f'{name}: {tlabel}, additive residuals')
    for ax in axes[i]:
        ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(11, 8), sharex='col')
for col, (name, raw, tr, tlabel, p) in enumerate(cases):
    dec = seasonal_decompose(tr, model='additive', period=p)
    for row, comp in enumerate(['observed', 'trend', 'seasonal', 'resid']):
        getattr(dec, comp).plot(ax=axes[row, col], lw=1)
        axes[row, col].set_ylabel(comp)
        axes[row, col].set_xlabel('')
    axes[0, col].set_title(f'{name}: {tlabel}, additive decomposition')
plt.tight_layout()
plt.show()

Commentary:

## 7. Seasonally adjusted series

*adjust on the right scale*

In [ ]:
cpi = tsdata.nigeria_cpi()
stl_cpi = STL(np.log(cpi), period=12, seasonal=13, robust=True).fit()
cpi_sa = np.exp(np.log(cpi) - stl_cpi.seasonal)
at_sa = np.exp(np.log(at) - stl_at.seasonal)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
at.plot(ax=axes[0], label='Original', lw=1)
at_sa.plot(ax=axes[0], label='seasonally adjusted', lw=1.5)
axes[0].set_title('Monthly austourists: Original vs Seasonally Adjusted')
axes[0].legend()
axes[0].set_xlabel('')
cpi.plot(ax=axes[1], label='Original', lw=1)
cpi_sa.plot(ax=axes[1], label='seasonally adjusted', lw=1.5)
axes[1].set_title('Nigeria CPI: Original vs Seasonally Adjusted')
axes[1].legend()
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()


Commentary:

## 8. Strength of trend and seasonality

*compute for both series and compare*

In [ ]:
def strength_features(s, period, log=False):
    y = np.log(s) if log else s
    r = STL(y, period=period, seasonal=13, robust=True).fit()
    ft = max(0, 1 - r.resid.var() / (r.trend + r.resid).var())
    fs = max(0, 1 - r.resid.var() / (r.seasonal + r.resid).var())
    return round(ft, 3), round(fs, 3)

rows = []
for name, s, lg in [('Austourists', at, True),
                    ('Nigeria CPI (sim.)', cpi, True),
                    ('White noise', pd.Series(np.random.default_rng(3).normal(), index=at.index), False)]:
    ft, fs = strength_features(s, 12, lg); rows.append((name, ft, fs))
print(pd.DataFrame(rows, columns=['series', 'trend strength', 'seasonal strength']).to_string(index=False))


In [ ]:
#Usig a bar chart to compare Trend and Seasonal Strength
series = ['Austourists', 'Nigeria CPI', 'White noise']
trend = [0.959, 1.000, 0.000]
seasonal = [0.912, 0.000, 0.268]

x = np.arange(len(series))
w = 0.35

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(x - w/2, trend, w, label='Trend strength')
ax.bar(x + w/2, seasonal, w, label='Seasonal strength')
ax.set_xticks(x)
ax.set_xticklabels(series)
ax.set_ylim(0, 1)
ax.set_title('Strength of trend and seasonality')
ax.legend()
plt.tight_layout()
plt.show()

Commentary:

## 9. Narratives (one page per series, also in the PDF report)

*what the series is doing; regularities; events or breaks; what a forecaster should worry about*

In [ ]:
# code


Commentary:

## AI-use statement

(Two to five lines: which tools, for what. 'No AI tools were used' is acceptable.)